In [1]:
!pip install requests pandas openpyxl


In [6]:
import requests, json

url = "https://dgidb.org/api/graphql"

query = """
{
  interactions(drugNames: ["Methotrexate"]) {
    edges {
      node {
        drug { name }
        gene { name }
        interactionTypes { type definition }
        sources { sourceDbName }
      }
    }
  }
}
"""

resp = requests.post(url, json={'query': query})
print("HTTP status:", resp.status_code)
print(json.dumps(resp.json(), indent=2))


HTTP status: 200
{
  "data": {
    "interactions": {
      "edges": [
        {
          "node": {
            "drug": {
              "name": "METHOTREXATE"
            },
            "gene": {
              "name": "FLT3"
            },
            "interactionTypes": [],
            "sources": [
              {
                "sourceDbName": "CKB-CORE"
              }
            ]
          }
        },
        {
          "node": {
            "drug": {
              "name": "METHOTREXATE"
            },
            "gene": {
              "name": "NFE2L2"
            },
            "interactionTypes": [],
            "sources": [
              {
                "sourceDbName": "DTC"
              }
            ]
          }
        },
        {
          "node": {
            "drug": {
              "name": "METHOTREXATE"
            },
            "gene": {
              "name": "SOD2"
            },
            "interactionTypes": [],
            "sources": [
              {


In [9]:

import requests
import pandas as pd

url = "https://dgidb.org/api/graphql"

# RA drugs
ra_drugs = ["Methotrexate", "Leflunomide", "Adalimumab", "Etanercept", "Tofacitinib"]

# Properly format the list for GraphQL - convert Python list to GraphQL list format
drugs_graphql_format = str(ra_drugs).replace("'", '"')  # Replace single quotes with double quotes for GraphQL

# Query for multiple drugs with properly formatted list
query = f"""
{{
  interactions(drugNames: {drugs_graphql_format}) {{
    edges {{
      node {{
        drug {{ name }}
        gene {{ name }}
        interactionTypes {{ type definition }}
        sources {{ sourceDbName }}
      }}
    }}
  }}
}}
"""

# Print the query for debugging (optional)
# print(query)

response = requests.post(url, json={'query': query})
data = response.json()

# Check if response contains data before proceeding
if "data" not in data:
    print("API Error:", data)  # Print the error response for debugging
    raise KeyError(f"API response doesn't contain 'data' key. Response: {data}")

# Parse results
rows = []
for edge in data["data"]["interactions"]["edges"]:
    node = edge["node"]
    rows.append({
        "Drug": node["drug"]["name"],
        "Gene": node["gene"]["name"],
        "Interaction_Type": "; ".join([it["type"] for it in node.get("interactionTypes", []) if "type" in it]),
        "Definition": "; ".join([it["definition"] for it in node.get("interactionTypes", []) if "definition" in it]),
        "Sources": "; ".join([s["sourceDbName"] for s in node.get("sources", []) if "sourceDbName" in s])
    })

# Create DataFrame and save
df = pd.DataFrame(rows)
df.to_excel("RA_Drug_Gene_Interactions.xlsx", index=False)

print("Saved:", len(df), "rows to RA_Drug_Gene_Interactions.xlsx")

Saved: 195 rows to RA_Drug_Gene_Interactions.xlsx


In [10]:
# ra_gene_to_drug.py
import requests, json, pandas as pd

url = "https://dgidb.org/api/graphql"

# RA genes of interest
ra_genes = ["TNF", "IL6R", "JAK1", "JAK3"]

# Properly quoted list
query = """
{
  interactions(geneNames: %s) {
    edges {
      node {
        drug { name }
        gene { name }
        interactionTypes { type definition }
        sources { sourceDbName }
      }
    }
  }
}
""" % json.dumps(ra_genes)

resp = requests.post(url, json={'query': query})
j = resp.json()

if 'data' not in j:
    print("Error:", json.dumps(j, indent=2))
    raise SystemExit

edges = j['data']['interactions']['edges']
rows = []
for edge in edges:
    node = edge['node']
    rows.append({
        "Gene": node['gene']['name'],
        "Drug": node['drug']['name'],
        "Interaction_Type": "; ".join([it.get('type','') for it in node['interactionTypes']]),
        "Definition": "; ".join([it.get('definition','') for it in node['interactionTypes']]),
        "Sources": "; ".join([s.get('sourceDbName','') for s in node['sources']])
    })

df = pd.DataFrame(rows)
df.to_excel("RA_Gene_Drug_Interactions.xlsx", index=False)
print("Saved", len(df), "rows to RA_Gene_Drug_Interactions.xlsx")


Saved 183 rows to RA_Gene_Drug_Interactions.xlsx


In [11]:
# ra_master_two_sheets.py
import requests, json, pandas as pd

url = "https://dgidb.org/api/graphql"

# RA drugs and genes
ra_drugs = ["Methotrexate", "Leflunomide", "Adalimumab", "Etanercept", "Tofacitinib"]
ra_genes = ["TNF", "IL6R", "JAK1", "JAK3"]

# --- Helper function ---
def run_query(query):
    resp = requests.post(url, json={'query': query}, timeout=30)
    j = resp.json()
    if 'data' not in j:
        print("Error from server:", json.dumps(j, indent=2))
        raise SystemExit
    return j

# --- Drug → Gene ---
query_drugs = """
{
  interactions(drugNames: %s) {
    edges {
      node {
        drug { name }
        gene { name }
        interactionTypes { type definition }
        sources { sourceDbName }
      }
    }
  }
}
""" % json.dumps(ra_drugs)

j_drugs = run_query(query_drugs)
edges = j_drugs['data']['interactions']['edges']

rows_drugs = []
for edge in edges:
    node = edge['node']
    rows_drugs.append({
        "Drug": node['drug']['name'],
        "Gene": node['gene']['name'],
        "Interaction_Type": "; ".join([it.get('type','') for it in node['interactionTypes']]),
        "Definition": "; ".join([it.get('definition','') for it in node['interactionTypes']]),
        "Sources": "; ".join([s.get('sourceDbName','') for s in node['sources']])
    })

df_drugs = pd.DataFrame(rows_drugs)

# --- Gene → Drug ---
query_genes = """
{
  interactions(geneNames: %s) {
    edges {
      node {
        drug { name }
        gene { name }
        interactionTypes { type definition }
        sources { sourceDbName }
      }
    }
  }
}
""" % json.dumps(ra_genes)

j_genes = run_query(query_genes)
edges = j_genes['data']['interactions']['edges']

rows_genes = []
for edge in edges:
    node = edge['node']
    rows_genes.append({
        "Gene": node['gene']['name'],
        "Drug": node['drug']['name'],
        "Interaction_Type": "; ".join([it.get('type','') for it in node['interactionTypes']]),
        "Definition": "; ".join([it.get('definition','') for it in node['interactionTypes']]),
        "Sources": "; ".join([s.get('sourceDbName','') for s in node['sources']])
    })

df_genes = pd.DataFrame(rows_genes)

# --- Save both into one Excel workbook ---
out_file = "RA_Drug_Gene_Interactions.xlsx"
with pd.ExcelWriter(out_file, engine="openpyxl") as writer:
    df_drugs.to_excel(writer, sheet_name="Drug_to_Gene", index=False)
    df_genes.to_excel(writer, sheet_name="Gene_to_Drug", index=False)

print(f"Done. Results saved to {out_file}")
print("Sheets: Drug_to_Gene, Gene_to_Drug")


Done. Results saved to RA_Drug_Gene_Interactions.xlsx
Sheets: Drug_to_Gene, Gene_to_Drug


In [12]:
import pandas as pd

# Read the Excel you already generated
df = pd.read_excel("RA_Drug_Gene_Interactions.xlsx", sheet_name="Drug_to_Gene")

# Save as CSV
df.to_csv("RA_Drug_Gene_Interactions.csv", index=False)
print("Saved CSV for Cytoscape import.")


Saved CSV for Cytoscape import.


In [14]:
import pandas as pd

# Load your Excel
df = pd.read_excel("RA_Drug_Gene_Interactions.xlsx", sheet_name="Drug_to_Gene")

# Export clean edge list with just 2 columns
edges = df[['Drug', 'Gene']]
edges.columns = ['source', 'target']

# Save as true CSV
edges.to_csv("RA_Drug_Gene_edges.csv", index=False, encoding="utf-8")
print("Created RA_Drug_Gene_edges.csv")


Created RA_Drug_Gene_edges.csv
